In [1]:
import math
import numpy as np 
import netCDF4 as nc
import xarray as xr
import thermofeel as tmf
import matplotlib.pyplot as plt
from scipy import ndimage

import zipfile
import os
import shutil
import gc
import cdsapi
from datetime import datetime

### define some functions that will be useful later

In [2]:
################################################################################
######################### FUNCTIONS TO CREATE A RELEVENT API REQUEST
##################################################################################

def number_of_days_in_month(month, year):
    month = int(month)
    year = int(year)
    
    # Days in each month
    if month in [1, 3, 5, 7, 8, 10, 12]:
        n_days = 31
    elif month in [4, 6, 9, 11]:
        n_days = 30
    else:  # February
        if (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0):
            n_days = 29
        else:
            n_days = 28
    
    # Return list of zero-padded strings
    return [f"{i:02d}" for i in range(1, n_days + 1)]


def needed_months(year):
    year = int(year)

    #define the staring month depending on the year

    if year in [2014, 2018]:
        start_month = 10
        end_month = 11
    else : 
        start_month = 1
        end_month = 12

    return [f"{i:02d}" for i in range (start_month, end_month + 1)]


################################################################################
######################### DOWNLOADING FUNCTIONS
##################################################################################


def download_MRT_data(y, m, days, A, path):
    dataset = "derived-utci-historical"
    request = {
        "variable": ["mean_radiant_temperature"],
        "version": "1_1",
        "product_type": "consolidated_dataset",
        "year": y,
        "month": m,
        "day": days,
        "data_format": "netcdf",
        "area": A
    }
    
    c = cdsapi.Client()
    c.retrieve(dataset, request, path)

def download_REAN_data(y, m, days, A, path):
    dataset = "reanalysis-era5-single-levels"
    request = {
        "product_type": ["reanalysis"],
        "variable": [
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "2m_dewpoint_temperature",
            "2m_temperature",
            "total_precipitation"
        ],
        "year": y,
        "month": m,
        "day": days,
        "time": [f"{h:02d}:00" for h in range(24)],
        "data_format": "netcdf",
        "download_format": "zip",
        "area": A
    }
    
    c= cdsapi.Client()
    c.retrieve(dataset, request, path)

    

#################################### THIS FUNCTION COMPUTES WINDSPEED, WET-BULB TEMPERATURE, GLOBE TEMPERATURE AND WET-BULB GLOBE TEMPERATURE
#################################### THE NEW VARIABLES ARE ADDED TO THE DATASET THAT IS PASSED AS ARGUMENT. 
#################################### /!\ THE FUNCTION MODIFIES THE DATASET. (nothing is deleted but new variables are added).
#################################### THE DATASET PASSED AS ARGUMENT SHOULD BE A XRARRAY WITH AT LEAST THE FOLLOWING VARIABLES:
####################################                  -THE TWO COMPONENTS OF WINDSPEED (named u10 and v10)
####################################                  -AIR TEMPERATURE (named t2m)
####################################                  -DEWPOINT TEMPERATURE (named d2m)
####################################                  -MEAN RADIANT TEMPERATURE (named mrt)


def compute_new_variables(dataset):
    # COMPUTE WINDSPEED
    u = dataset['u10']
    v = dataset['v10']
    
    wind_speed = np.sqrt(u**2 + v**2)
    
    # Add it to the dataset
    dataset['wind_speed'] = wind_speed

    #COMPUTE WBT
    #first compute relative humidity using T and Td
    t2_k = dataset['t2m']
    d2_k = dataset['d2m']
    
    dataset['rh'] = tmf.calculate_relative_humidity_percent(t2_k, d2_k)
    
    #then compute wet-bulb temp using T and RH
    rh = dataset['rh']
    
    dataset['wbt'] = tmf.calculate_wbt(t2_k, rh)

    #COMPUTE BGT
    mrt = dataset['mrt']
    
    dataset['bgt'] = tmf.calculate_bgt(t2_k, mrt, wind_speed)


    #COMPUTE WBGT
    wb_k = dataset['wbt']
    gt_k = dataset['bgt']
    
    dataset['wbgt'] = 0.7 * wb_k + 0.2 * gt_k + 0.1 * t2_k


################################################################################
######################### FUNCTION TO DELETE THE TEMPORARY FILES
##################################################################################


def delete_files(folder):
    for filename in os.listdir(folder):
        file_path = os.path.join(folder, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.remove(file_path)  # remove file or symlink
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)  # remove subfolder
        except Exception as e:
            print(f'Failed to delete {file_path}. Reason: {e}')
    print(f"Deleted all contents of folder: {folder}")

### Enter the parameters that are needed to scrap the ERA5 data

In [4]:
# the folder in which the zip files are downnloaded
download_folder = './interm/zip_files/025_files'
#the folder in which the data are saved
data_folder = './output/'

#the years needed
years = ['2014', '2015', '2016',  '2018', '2019']

#the coordinates needed. Carreful : the coordinates should be rounded at the (upper or lower) 0.25. For instance : 35.6->35.5
India_area = [35.5, 68.0, 6.5, 97.5]

### The following loop downloads hourly data from the ERA5 API. Then computes the needed climate variables and stores the results as netcdf files in 'data_folder'.

In [5]:
start = datetime.now()
for y in years:
    #get the required months
    months = needed_months(y)
    
    for m in months:
        start1 = datetime.now()
        #get the days corresponding to month m and year y
        needed_days = number_of_days_in_month(m, y)
    
        ################################################################# DOWNLOAD THE REANALYSIS DATA
        #check if the data already exist for this year-month. if no, download them.
        filename_rean = 'REAN_' + y + m + '.zip'
        filepath = os.path.join(download_folder, filename_rean)
        if os.path.exists(filepath):
            print(f"File {filename_rean} already exists")
        else:
            print(f"File {filename_rean} does not exist in the demanded folder. Proceed to download.")
            download_REAN_data(y, m, needed_days, India_area, filepath)
            print(f"File {filename_rean} downloaded")
        
        ################################################################# DOWNLOAD THE MRT DATA
        #check if the data already exist for this year-month. if no, download them.
        filename_mrt = 'MRT_' + y + m + '.zip'
        filepath = os.path.join(download_folder, filename_mrt)
        if os.path.exists(filepath):
            print(f"File {filename_mrt} already exists")
        else:
            print(f"File {filename_mrt} does not exist in the demanded folder. Proceed to download.")
            download_MRT_data(y, m, needed_days, India_area, filepath)
            print(f"File {filename_mrt} downloaded")


        
        
        ########################################################### UNZIP, OPEN AND STORE ALL THE DATA INTO A UNIQUE XRARRAY OBJECT CALLED combined
        
        # Path to MRT zipped archive
        zip_path = os.path.join(download_folder, filename_mrt)
        extract_dir = 'interm/unzipped_MRT'
        
        # Extract all files
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        
        # Collect all .nc file paths
        nc_files = [os.path.join(extract_dir, f) for f in os.listdir(extract_dir) if f.endswith('.nc')]
        
        # Open and concatenate along time
        MRT = xr.open_mfdataset(nc_files, combine='by_coords').load()
        MRT.close()
        
        #Path to reanalysis zipped archive
        zip_path = os.path.join(download_folder, filename_rean)
        extract_dir = 'interm/unzipped_REAN'
        
        #unzip the REAN file
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        
        #collect the two netcdf from the REAN unzipped archive
        REAN = xr.open_dataset(os.path.join(extract_dir, 'data_stream-oper_stepType-instant.nc'), engine = "h5netcdf").load()
        REAN.close()
        precip = xr.open_dataset(os.path.join(extract_dir, 'data_stream-oper_stepType-accum.nc'), engine = "h5netcdf").load()
        precip.close()
        #merge the two
        REAN = xr.merge([REAN, precip])

        #rename the REAN dimensions
        REAN = REAN.rename({
            "valid_time": "time",
            "latitude": "lat",
            "longitude": "lon"
        })

        # merge REAN AND MRT
        combined = xr.merge([REAN, MRT])
        

        ################################################################# COMPUTE THE NEW VARIABLES
        
        compute_new_variables(combined)
        
    
        ################################################################# SAVE THE FINAL NETCDF FOR THE CURRENT MONTH
    
        vars_to_keep = ['t2m', 'rh', 'wbt', 'mrt', 'bgt', 'wbgt', 'tp']
        subset = combined[vars_to_keep]
        
        output_path = os.path.join(data_folder, '025_alltemps_' + y + m + '.nc')
        subset.to_netcdf(output_path)

        print('File ' + '025_alltemps_' + y + m + '.nc' + ' saved')    



        
        ################################################################# CLEAN EVERYTHING FOR THE NEXT STEP
        
        #del REAN, precip, MRT, combined, subset
        #empty the extract folders
        extract_dir = ['interm/unzipped_MRT', 'interm/unzipped_REAN']
        for folder in extract_dir:
            delete_files(folder)
        end = datetime.now()
        print(f'this stage took {end-start1}')
        print(f'total time since entering the loop: {end-start}')
        print('-------------------------------------------------')

File REAN_201410.zip already exists
File MRT_201410.zip already exists
File 025_alltemps_201410.nc saved
Deleted all contents of folder: interm/unzipped_MRT
Deleted all contents of folder: interm/unzipped_REAN
this stage took 0:00:06.948146
total time since entering the loop: 0:00:06.948503
-------------------------------------------------
File REAN_201411.zip already exists
File MRT_201411.zip already exists
File 025_alltemps_201411.nc saved
Deleted all contents of folder: interm/unzipped_MRT
Deleted all contents of folder: interm/unzipped_REAN
this stage took 0:00:05.206822
total time since entering the loop: 0:00:12.155363
-------------------------------------------------
File REAN_201501.zip already exists
File MRT_201501.zip already exists
File 025_alltemps_201501.nc saved
Deleted all contents of folder: interm/unzipped_MRT
Deleted all contents of folder: interm/unzipped_REAN
this stage took 0:00:05.119847
total time since entering the loop: 0:00:17.275265
------------------------